# 04 — Lens Distortion Correction

**Objective:** Load the calibration profile saved in Notebook 03 and apply lens undistortion to images. Visualise the correction effect and understand the alpha trade-off.

## The alpha parameter

After undistorting, some pixels near the image border become invalid (mapped outside the sensor). `cv2.getOptimalNewCameraMatrix` computes a new camera matrix that controls how much of the undistorted image is kept:

| alpha | Effect |
|---|---|
| **0.0** | Crop all invalid pixels — no black borders, but valid area is smaller |
| **1.0** | Keep all pixels — black borders visible at edges |
| **0.5** | Balanced — recommended default |


In [ ]:
import sys, logging
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.profile import CalibrationProfile
from calibvision.dewarper import LensDewarper
from calibvision.utils import side_by_side

logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

PROJECT_ROOT = Path('..').resolve()
PROFILE_PATH = PROJECT_ROOT / 'configs' / 'saved_profiles' / 'synthetic_demo.yaml'
CALIB_DIR    = PROJECT_ROOT / 'data' / 'calibration_images'
print('Setup complete')

In [ ]:
# ── 4.1  Load profile and inspect ────────────────────────────────────────────
profile = CalibrationProfile.load(PROFILE_PATH)
print(profile.summary())

In [ ]:
# ── 4.2  Initialise LensDewarper (precomputes remap tables) ──────────────────
dewarper = LensDewarper(profile, alpha=0.5)
print(f'Valid ROI after undistortion: {dewarper.valid_roi}')

In [ ]:
# ── 4.3  Apply to calibration images and compare ─────────────────────────────
sample_paths = sorted(CALIB_DIR.glob('*.png'))[:4]

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(14, len(sample_paths) * 3.5))
if len(sample_paths) == 1:
    axes = [axes]

for i, path in enumerate(sample_paths):
    orig = cv2.imread(str(path))
    undist = dewarper.undistort(orig)

    axes[i][0].imshow(cv2.cvtColor(orig,   cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'Original — {path.name}', fontsize=8)
    axes[i][0].axis('off')

    axes[i][1].imshow(cv2.cvtColor(undist, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title('Undistorted (alpha=0.5)', fontsize=8)
    axes[i][1].axis('off')

plt.suptitle('Lens Distortion Correction — Original vs Undistorted', fontsize=11)
plt.tight_layout()
plt.show()
print(f'Processed {len(sample_paths)} images')

In [ ]:
# ── 4.4  Alpha comparison: 0.0 vs 0.5 vs 1.0 ─────────────────────────────────
test_img = cv2.imread(str(sorted(CALIB_DIR.glob('*.png'))[0]))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, alpha in zip(axes, [0.0, 0.5, 1.0]):
    dw = LensDewarper(profile, alpha=alpha)
    result_img = dw.undistort(test_img)
    ax.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'alpha={alpha}\nROI: {dw.valid_roi}', fontsize=9)
    ax.axis('off')

plt.suptitle('Alpha parameter effect: 0.0 (crop) → 0.5 (balanced) → 1.0 (keep all)')
plt.tight_layout()
plt.show()

print('Note: For synthetic images the correction is minimal (near-zero distortion).')
print('With a real camera the barrel/pincushion effect will be clearly visible.')

In [ ]:
# ── 4.5  Pixel-level difference map ─────────────────────────────────────────
# Even for synthetic images, the remap introduces slight differences.
orig_gray  = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY).astype(np.float32)
undist_05  = LensDewarper(profile, alpha=0.5).undistort(test_img)
undist_gray = cv2.cvtColor(undist_05, cv2.COLOR_BGR2GRAY).astype(np.float32)

diff = np.abs(orig_gray - undist_gray)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
im = ax1.imshow(diff, cmap='hot')
ax1.set_title('Pixel difference |original − undistorted|')
ax1.axis('off')
plt.colorbar(im, ax=ax1, label='Pixel intensity difference')

ax2.hist(diff.ravel(), bins=50, color='#3498db', edgecolor='white', log=True)
ax2.set_xlabel('Pixel difference')
ax2.set_ylabel('Count (log scale)')
ax2.set_title(f'Difference distribution\nMean={diff.mean():.2f}  Max={diff.max():.2f}')

plt.tight_layout()
plt.show()

## Summary

| Step | Result |
|---|---|
| Profile loaded | ✓ from `configs/saved_profiles/synthetic_demo.yaml` |
| Remap tables precomputed | ✓ (single init, fast batch processing) |
| Alpha comparison | ✓ |
| Difference map | ✓ |

## Limitation

Synthetic calibration images have near-zero lens distortion by design. The visual correction here will be subtle. With a real camera (especially wide-angle), barrel distortion is clearly visible.

## Next step

Open `05_perspective_rectification.ipynb` to apply four-point homography-based perspective correction.
